## Assignment: Image recognition
- Alumno 1: Aritz Bermejo Canal
- Alumno 2: Guillermo Alonso Rello
- Alumno 3: Juan García Santos

The goals of the assignment are:
* Develop proficiency in using Tensorflow/Keras for training Neural Nets (NNs).
* Put into practice the acquired knowledge to optimize the parameters and architecture of a feedforward Neural Net (ffNN), in the context of an image recognition problem.
* Put into practice NNs specially conceived for analysing images. Design and optimize the parameters of a Convolutional Neural Net (CNN) to deal with previous task.
* Train popular architectures from scratch (e.g., GoogLeNet, VGG, ResNet, ...), and compare the results with the ones provided by their pre-trained versions using transfer learning.

Información sobre el dataset:
El dataset original son 846 imagenes tomadas por un satélite de diferentes lugares del mundo. 761 imágenes son para train y 85 para test. En ellas aparecen objetos de 60 categorías. Se anotó con rectángulos la posición donde aparece un objeto en cada una de las imágenes mediante el punto (x,y) y la altura y la anchura el rectángulo.
Nosotros no vamos a trabajar con las imágenes originales, sino con los recortes de todos los objetos presentes en ellas. Por cada rectángulo posicionado encima de cada objeto se ha creado una imagen nueva de tamaño 224x224(x3 por el RGB). Esto significa que las imagenes no tienen por qué guardar las proporciones exactas del objeto original (se ha podido achatar o ampliar). Los recortes que pertenecen al conjunto de train son aquellos que vienen de las imágenes del conjunto de train y lo mismo para el test. Las 60 categorías están muy desbalancedas, por lo que solo vamos a utilizar los recortes de las 12 categorías más comunes.
Follow the link below to download the classification data set  “xview_recognition”: [https://drive.upm.es/s/4oNHlRFEd71HXp4](https://drive.upm.es/s/4oNHlRFEd71HXp4)
Los archivos del dataset son 2 JSON y 2 carpetas de imágenes, 1 de cada para train y test. El JSON se usa a modo de índice, esencialmente contiene pares (nombre_imagen, categoría). La carpeta de imágenes tiene 1 subcarpeta por categoría de objetos, donde se encuentran todas las imágenes de objetos de ese tipo, cada una con un nombre_imagen único referenciado desde el JSON.

First challenge: use of ffNN. We need to decide:
* Number of layers and number of units in each layer of your NN.
* Optimization algorithm and parameters to train the network.
* Check the evolution of these parameters during the optimization by using a validation subset and decide when to stop training (note that the test data set can only be used to evaluate the model).

You must produce a few slides in PDF format describing the problem, all architectures, the performance obtained with your model on the train/valid/test data sets, plots of the evolution of costs and classification performance. Describe also the process that you have followed to reach your solution.

Estrategia de ejecuciones:
1º Decidir el tamaño de la red: número de capas ocultas y neuronas por capa. Probar ascendentemente hasta llegar a un máximo de 5 capas. En principio el número de neuronas debe ir decreciendo capa a capa.
Para ello utilizar inicialmente Adam, RELU y descubrir un buen learning rate. Además, utilizar siempre regularización L2 (o dropout, solo 1 a la vez [aunque en principio no se puede dropout]) + batch normalization + early stopping con el conjunto de validación. También hay que decidir el tamaño de este conjunto, en principio conviene el mismo que el de test (los profes han puesto 10%). Prestar especial atención a la clase helicóptero, que solo tiene 70, podría pasar que la dividir en train y validación no se repartan bien.
2º Una vez estimado el tamaño, decidir mejor algoritmo (probar los otros que no son ADAM), con sus hiperparámetros, y función de activación (probar variantes de RELU). Finalmente probar a usar bagging.


In [1]:
import uuid
import numpy as np

class GenericObject:
    """
    Generic object data.
    """
    def __init__(self):
        self.id = uuid.uuid4()
        self.bb = (-1, -1, -1, -1)
        self.category= -1
        self.score = -1

class GenericImage:
    """
    Generic image data.
    """
    def __init__(self, filename):
        self.filename = filename
        self.tile = np.array([-1, -1, -1, -1])  # (pt_x, pt_y, pt_x+width, pt_y+height)
        self.objects = list([]) # Realmente solo hay 1 objeto por cada imagen

    def add_object(self, obj: GenericObject):
        self.objects.append(obj)

In [2]:
categories = {0: 'Cargo plane', 1: 'Helicopter', 2: 'Small car', 3: 'Bus', 4: 'Truck', 5: 'Motorboat', 6: 'Fishing vessel', 7: 'Dump truck', 8: 'Excavator', 9: 'Building', 10: 'Storage tank', 11: 'Shipping container'}

In [3]:
import warnings
import rasterio
import numpy as np

# Devuelve la matriz 3D con los bits de una imagen dado el nombre del archivo
def load_geoimage(filename):
    warnings.filterwarnings('ignore', category=rasterio.errors.NotGeoreferencedWarning)
    src_raster = rasterio.open(filename, 'r')
    # RasterIO to OpenCV (see inconsistencies between libjpeg and libjpeg-turbo)
    input_type = src_raster.profile['dtype']
    input_channels = src_raster.count # 3 canales (RGB)
    img = np.zeros((src_raster.height, src_raster.width, src_raster.count), dtype=input_type)
    for band in range(input_channels): # Rellenar toda la matriz 2D del canal k de la matriz
        img[:, :, band] = src_raster.read(band+1)
    return img

# Genera grupos de imágenes del tamaño del batch durante todo el proceso de entrenamiento
def generator_images(objs, batch_size, batch_size_aug, train_dataset_size, do_shuffle=False):
    while True:
        if do_shuffle:
            np.random.shuffle(objs)
        groups = [objs[i:i+batch_size] for i in range(0, len(objs), batch_size)]
        for group in groups: # 1 epoch
            images, labels = [], [] # Matrices de imagen y one-hot encoding de cada una del grupo
            for (filename, obj) in group: # 1 mini-batch
                # Load image
                images.append(load_geoimage(filename)) # Bits de la imagen
                probabilities = np.zeros(len(categories)) # Vector para one-hot enconding
                probabilities[list(categories.values()).index(obj.category)] = 1
                labels.append(probabilities)
            images = np.array(images).astype(np.float32)
            labels = np.array(labels).astype(np.float32)
            yield images, labels # Devolver todo el grupo
        if(batch_size < train_dataset_size/10): # AL final de la epoch
            batch_size = batch_size * batch_size_aug
            print(batch_size)

In [4]:
#### Funciones de train

import math
import datetime
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, Activation, Flatten, BatchNormalization
from tensorflow.keras.regularizers import L2

from tensorflow.keras.optimizers import Adam, SGD
from tensorflow.keras.callbacks import TerminateOnNaN, EarlyStopping, ReduceLROnPlateau, ModelCheckpoint, TensorBoard

def print_params(date_time, l_rate, b_size, bs_aug_factor, epochs, optimizer, opt_hparams, neurons_hidden_layers, activ_fun, regularization, reg_param, batch_norm: bool):
    print('Model parameters')
    print('Datetime', date_time)
    if isinstance(l_rate, float):
        print('Learning rate (fijo)', l_rate)
    else:
        print('Cosine decay')
        print('Learning rate antes del warmup', l_rate.initial_learning_rate)
        print('Numero de steps en warnup', l_rate.warmup_steps)
        print('Learning rate después del warmup', l_rate.warmup_target) 
        print('Numero de decay steps coseno', l_rate.decay_steps)
        print('Reducción máxima con decay coseno', l_rate.alpha)
    print('Batch size', b_size)
    print('Batch size augmentation', bs_aug_factor)
    print('Epochs', epochs)
    print('Optimizer', optimizer)
    print('Optimizer hiperparams', opt_hparams)
    print('Capas ocultas', neurons_hidden_layers)
    print('Función de activación', activ_fun)
    print('Regularización', regularization)
    print('Reg param', reg_param)
    print('Normalización batch', batch_norm)
    
"""
Función para generar red neuronal. Se puede elegir:
- Learning rate o decreasing schedule
- Batch size
- Factor de incremento del batch size (cuanto cambia por epoch)
- Max Epochs
- Optimizer: Adam, SGD con momento o momento de Nesterov
- Hiperparámetros específicos del optimizador (en un array)
- Número de neuronas en cada capa oculta (un array de tamaño número de capas ocultas)
- Función de activación en cada capa oculta (la misma en todas)
- Uso de regularización (L2 o Dropout)
- Uso de batch norm (Sí o No)
"""
def train_net(l_rate, b_size, bs_aug_factor, epochs, optimizer, opt_hparams, neurons_hidden_layers, activ_fun, regularization, reg_param, batch_norm: bool):
    model = Sequential()
    # Capa de entrada, una neurona por píxel
    model.add(Flatten(input_shape=(224, 224, 3))) 
    if(batch_norm): # No puede tener regularización, solo batch norm
        model.add(BatchNormalization())
    model.add(Activation(activ_fun))

    # Añadir capas intermedias
    for n in neurons_hidden_layers:
        if(regularization):
            model.add(Dense(n, kernel_regularizer=L2(reg_param))) # Capa oculta
        else:
            model.add(Dense(n)) # Capa oculta
        if(batch_norm):
            model.add(BatchNormalization())
        model.add(Activation(activ_fun))

    # Capa final, una salida por categoría (12)
    model.add(Dense(len(categories)))
    model.add(Activation('softmax'))
    
    model.summary()

    # Use optimizer
    if(optimizer == 'Adam'):
        opt = Adam(learning_rate=l_rate, beta_1=opt_hparams['beta1'], beta_2=opt_hparams['beta2'], epsilon=opt_hparams['epsilon'], amsgrad=True, clipnorm=1.0, clipvalue=0.5)
    elif(optimizer == 'SGD'):
        opt = SGD(learning_rate=l_rate, momentum=opt_hparams['momentum'], nesterov=opt_hparams['nesterov'])
    
    model.compile(optimizer=opt, loss='categorical_crossentropy', metrics=['accuracy'])

    # Callbacks
    model_checkpoint = ModelCheckpoint('model.hdf5', monitor='val_accuracy', verbose=1, save_best_only=True)
    reduce_lr = ReduceLROnPlateau('val_accuracy', factor=0.1, patience=10, verbose=1) # Redicir el learning rate si se estanca
    early_stop = EarlyStopping('val_accuracy', patience=40, verbose=1)
    terminate = TerminateOnNaN()    
    
    current_time = datetime.datetime.now()
    time_string = current_time.strftime("%Y-%m-%d_%H-%M-%S")  # Replace colons with underscores
    tensorboard = TensorBoard(log_dir=f"./logs/{time_string}")
    
    callbacks = [model_checkpoint, reduce_lr, early_stop, terminate, tensorboard]

    # Preparar los datos y los generadores según el batch size
    objs_train = [(ann.filename, obj) for ann in anns_train for obj in ann.objects]
    objs_valid = [(ann.filename, obj) for ann in anns_valid for obj in ann.objects]
    train_steps = math.ceil(len(objs_train)/b_size)
    valid_steps = math.ceil(len(objs_valid)/b_size)
    train_generator = generator_images(objs_train, b_size, bs_aug_factor, len(objs_train), do_shuffle=True) # randomizar para no sesgar batches con orden
    valid_generator = generator_images(objs_valid, b_size, bs_aug_factor, len(objs_train), do_shuffle=False) # no es necesario randomizar, solo se testean

    print_params(time_string, l_rate, b_size, bs_aug_factor, epochs, optimizer, opt_hparams, neurons_hidden_layers, activ_fun, regularization, reg_param, batch_norm)
    # TODO ACTUALIZAR BATCH SIZE
    #for epoch in range(epochs): # 1 vez por época para poder actualizar el batch size
    h = model.fit(train_generator, validation_data=valid_generator, steps_per_epoch=train_steps, validation_steps=valid_steps, epochs=epochs, callbacks=callbacks, verbose=1)
    """if(b_size < len(objs_train)/10): # Reducir el batch size hasta llegar a un límite en función del tamaño del dataset de entrenamiento
        b_size = b_size * bs_aug_factor
        train_steps = math.ceil(len(objs_train)/b_size)
        valid_steps = math.ceil(len(objs_valid)/b_size)
        print(b_size)"""

    return model, h

2023-10-23 00:00:37.568118: E tensorflow/compiler/xla/stream_executor/cuda/cuda_dnn.cc:9342] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2023-10-23 00:00:37.568145: E tensorflow/compiler/xla/stream_executor/cuda/cuda_fft.cc:609] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2023-10-23 00:00:37.568167: E tensorflow/compiler/xla/stream_executor/cuda/cuda_blas.cc:1518] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2023-10-23 00:00:37.574436: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


In [5]:
#### Funciones de test

import matplotlib.pyplot as plt
import numpy as np

def test_net(model, test_anns):
    y_true, y_pred = [], []
    for ann in test_anns:
        # Load image
        image = load_geoimage(ann.filename)
        for obj_pred in ann.objects:
            # Generate prediction
            warped_image = np.expand_dims(image, 0)
            predictions = model.predict(warped_image)
            # Save prediction
            pred_category = list(categories.values())[np.argmax(predictions)]
            pred_score = np.max(predictions)
            y_true.append(obj_pred.category)
            y_pred.append(pred_category)
    return y_true, y_pred

def draw_confusion_matrix(cm, categories):
    # Draw confusion matrix
    fig = plt.figure(figsize=[6.4*pow(len(categories), 0.5), 4.8*pow(len(categories), 0.5)])
    ax = fig.add_subplot(111)
    cm = cm.astype('float') / np.maximum(cm.sum(axis=1)[:, np.newaxis], np.finfo(np.float64).eps)
    im = ax.imshow(cm, interpolation='nearest', cmap=plt.cm.get_cmap('Blues'))
    ax.figure.colorbar(im, ax=ax)
    ax.set(xticks=np.arange(cm.shape[1]), yticks=np.arange(cm.shape[0]), xticklabels=list(categories.values()), yticklabels=list(categories.values()), ylabel='Annotation', xlabel='Prediction')
    # Rotate the tick labels and set their alignment
    plt.setp(ax.get_xticklabels(), rotation=45, ha="right", rotation_mode="anchor")
    # Loop over data dimensions and create text annotations
    thresh = cm.max() / 2.0
    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            ax.text(j, i, format(cm[i, j], '.2f'), ha="center", va="center", color="white" if cm[i, j] > thresh else "black", fontsize=int(20-pow(len(categories), 0.5)))
    fig.tight_layout()
    plt.show(fig)

# Imprimir resultados a partir de la matriz de confusión
def print_results(cm):
    # Compute the accuracy
    correct_samples_class = np.diag(cm).astype(float)
    total_samples_class = np.sum(cm, axis=1).astype(float)
    total_predicts_class = np.sum(cm, axis=0).astype(float)
    print('Mean Accuracy: %.3f%%' % (np.sum(correct_samples_class) / np.sum(total_samples_class) * 100))
    acc = correct_samples_class / np.maximum(total_samples_class, np.finfo(np.float64).eps)
    print('Mean Recall: %.3f%%' % (acc.mean() * 100))
    acc = correct_samples_class / np.maximum(total_predicts_class, np.finfo(np.float64).eps)
    print('Mean Precision: %.3f%%' % (acc.mean() * 100))
    for idx in range(len(categories)):
        # True/False Positives (TP/FP) refer to the number of predicted positives that were correct/incorrect.
        # True/False Negatives (TN/FN) refer to the number of predicted negatives that were correct/incorrect.
        tp = cm[idx, idx]
        fp = sum(cm[:, idx]) - tp
        fn = sum(cm[idx, :]) - tp
        tn = sum(np.delete(sum(cm) - cm[idx, :], idx))
        # True Positive Rate: proportion of real positive cases that were correctly predicted as positive.
        recall = tp / np.maximum(tp+fn, np.finfo(np.float64).eps)
        # Precision: proportion of predicted positive cases that were truly real positives.
        precision = tp / np.maximum(tp+fp, np.finfo(np.float64).eps)
        # True Negative Rate: proportion of real negative cases that were correctly predicted as negative.
        specificity = tn / np.maximum(tn+fp, np.finfo(np.float64).eps)
        # Dice coefficient refers to two times the intersection of two sets divided by the sum of their areas.
        # Dice = 2 |A∩B| / (|A|+|B|) = 2 TP / (2 TP + FP + FN)
        f1_score = 2 * ((precision * recall) / np.maximum(precision+recall, np.finfo(np.float64).eps))
        print('> %s: Recall: %.3f%% Precision: %.3f%% Specificity: %.3f%% Dice: %.3f%%' % (list(categories.values())[idx], recall*100, precision*100, specificity*100, f1_score*100))

In [6]:
#### Funciones para cargar BD
import json
import numpy as np

# Load JSON and then use it to get objects
def import_database(json_file):
    # Load database JSON.
    with open(json_file) as ifs:
        json_data = json.load(ifs)
    ifs.close()

    counts = dict.fromkeys(categories.values(), 0)
    anns = []
    # Crear un GenericImage por cada archivo con su GenericObject correspondiente categorizado dentro
    for json_img, json_ann in zip(json_data['images'].values(), json_data['annotations'].values()):
        image = GenericImage(json_img['filename'])
        image.tile = np.array([0, 0, json_img['width'], json_img['height']])
        obj = GenericObject()
        obj.bb = (int(json_ann['bbox'][0]), int(json_ann['bbox'][1]), int(json_ann['bbox'][2]), int(json_ann['bbox'][3]))
        obj.category = json_ann['category_id']
        # Resampling strategy to reduce training time
        counts[obj.category] += 1
        image.add_object(obj)
        anns.append(image)
    print(counts)

    return anns

In [7]:
# Cargamos las BD de train y test

json_train_file = 'xview_ann_train.json'
json_test_file = 'xview_ann_test.json'
print('Train')
anns_train = import_database(json_train_file)
print('Test')
anns_test = import_database(json_test_file)

Train
{'Cargo plane': 635, 'Helicopter': 70, 'Small car': 4290, 'Bus': 2155, 'Truck': 2746, 'Motorboat': 1069, 'Fishing vessel': 706, 'Dump truck': 1236, 'Excavator': 789, 'Building': 4689, 'Storage tank': 1469, 'Shipping container': 1523}
Test
{'Cargo plane': 83, 'Helicopter': 1, 'Small car': 487, 'Bus': 242, 'Truck': 305, 'Motorboat': 394, 'Fishing vessel': 93, 'Dump truck': 122, 'Excavator': 57, 'Building': 542, 'Storage tank': 243, 'Shipping container': 66}


In [8]:
# Separar conjunto de train
from sklearn.model_selection import train_test_split
anns_train, anns_valid = train_test_split(anns_train, test_size=0.1, random_state=1, shuffle=True)
N_train = len(anns_train) # Size of train dataset (without valid)

In [9]:
# Crea un schedule de cosine decay

def get_lr_schedule(N_steps, percentage_warmup, initial_learning_rate, target_learning_rate, reduction_factor):
    warmup_steps = math.floor(percentage_warmup * N_steps) # Step = minibatch
    decay_steps = math.floor((1 - percentage_warmup) * N_steps) # Pasos de decay coseno
    l_rate_decay = CosineDecay(
        initial_learning_rate, decay_steps, alpha=reduction_factor, 
        warmup_target=target_learning_rate, warmup_steps=warmup_steps
    )
    print('Warmup', warmup_steps, 'Decay', decay_steps)
    return l_rate_decay
    

In [10]:
# Permite crear varias redes haciendo listas de parametros
from tensorflow.keras.optimizers.schedules import CosineDecay
import math
import numpy as np
from tensorflow import random

random.set_seed(120)

# Definir el cosine decay

percentage_warmup = 0.1
initial_learning_rate = 0.001 # Antes del warmup
target_learning_rate = 0.05 # Después del warmup
reduction_factor = 1e-3 # Factor de cuanto va a decrecer

cosine_decay_params = [percentage_warmup, initial_learning_rate, target_learning_rate, reduction_factor]

num_models = 30

b_size = [16, 32, 64]
b_size = [b for b in b_size for _ in range(num_models//3)]
N_epochs = [30]
N_epochs = [e for e in N_epochs for _ in range(num_models)]
l_rate = [0.01, cosine_decay_params] # Si tiene un número: lr inicial que no decrece, si lista: parametros del cosine decay
l_rate = [lr for lr in l_rate for _ in range(num_models//2)]

bs_aug_factor = [1] * num_models # Dejarlo igual
optimizer = ['Adam'] + ['SGD'] * 4
optimizer = optimizer * (num_models//5)
opt_hparams = [
                {'beta1':0.9, 'beta2':0.999, 'epsilon':1e-8}, 
               {'momentum':0.2, 'nesterov':False},
                {'momentum':0.5, 'nesterov':False},
                {'momentum':0.2, 'nesterov':True},
                {'momentum':0.5, 'nesterov':True},
              ] 
opt_hparams = opt_hparams * (num_models//5)
neurons_hidden_layers = [[70, 25]]
neurons_hidden_layers = neurons_hidden_layers * num_models
activ_fun = ['relu'] * num_models
regularization = [True] * num_models
reg_param = [0.01] * num_models
batch_norm = [True] * num_models

In [ ]:
import math
from sklearn.metrics import confusion_matrix

for i in range(num_models):
    N_steps = (N_train/b_size[i]) * N_epochs[i]
    if isinstance(l_rate[i], list): # Si es cosine decay
        lr = get_lr_schedule(N_steps, l_rate[i][0], l_rate[i][1], l_rate[i][2], l_rate[i][3])
    else:
        lr = l_rate[i]
    model, h = train_net(lr, b_size[i], bs_aug_factor[i], N_epochs[i], optimizer[i], opt_hparams[i], 
                         neurons_hidden_layers[i], activ_fun[i], regularization[i], reg_param[i], batch_norm[i])
    
    # Best validation model
    best_idx = int(np.argmax(h.history['val_accuracy']))
    best_value = np.max(h.history['val_accuracy'])
    print('Best validation model: epoch ' + str(best_idx+1), ' - val_accuracy ' + str(best_value))
    
    y_true, y_pred = test_net(model, anns_test)
    # Compute the confusion matrix
    cm = confusion_matrix(y_true, y_pred, labels=list(categories.values()))
    draw_confusion_matrix(cm, categories)
    print_results(cm)

Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 flatten (Flatten)           (None, 150528)            0         
                                                                 
 batch_normalization (Batch  (None, 150528)            602112    
 Normalization)                                                  
                                                                 
 activation (Activation)     (None, 150528)            0         
                                                                 
 dense (Dense)               (None, 70)                10537030  
                                                                 
 batch_normalization_1 (Bat  (None, 70)                280       
 chNormalization)                                                
                                                                 
 activation_1 (Activation)   (None, 70)                0

2023-10-23 00:00:40.890720: I tensorflow/compiler/xla/stream_executor/cuda/cuda_gpu_executor.cc:894] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero. See more at https://github.com/torvalds/linux/blob/v6.0/Documentation/ABI/testing/sysfs-bus-pci#L344-L355
2023-10-23 00:00:40.917314: I tensorflow/compiler/xla/stream_executor/cuda/cuda_gpu_executor.cc:894] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero. See more at https://github.com/torvalds/linux/blob/v6.0/Documentation/ABI/testing/sysfs-bus-pci#L344-L355
2023-10-23 00:00:40.917527: I tensorflow/compiler/xla/stream_executor/cuda/cuda_gpu_executor.cc:894] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero. See more at https://github.com/torvalds/linux/blob/v6.0/Documentation/ABI/testing/sysf

Trainable params: 10840363 (41.35 MB)
Non-trainable params: 301246 (1.15 MB)
_________________________________________________________________
Model parameters
Datetime 2023-10-23_00-00-41
Learning rate (fijo) 0.01
Batch size 16
Batch size augmentation 1
Epochs 30
Optimizer Adam
Optimizer hiperparams {'beta1': 0.9, 'beta2': 0.999, 'epsilon': 1e-08}
Capas ocultas [70, 25]
Función de activación relu
Regularización True
Reg param 0.01
Normalización batch True


2023-10-23 00:00:41.740071: I tensorflow/tsl/platform/default/subprocess.cc:304] Start cannot spawn child process: No such file or directory


Epoch 1/30


2023-10-23 00:00:45.291889: I tensorflow/compiler/xla/service/service.cc:168] XLA service 0x7f04cc548360 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
2023-10-23 00:00:45.291912: I tensorflow/compiler/xla/service/service.cc:176]   StreamExecutor device (0): NVIDIA GeForce GTX 1650 Ti, Compute Capability 7.5
2023-10-23 00:00:45.299304: I tensorflow/compiler/mlir/tensorflow/utils/dump_mlir_util.cc:269] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.
2023-10-23 00:00:45.322136: I tensorflow/compiler/xla/stream_executor/cuda/cuda_dnn.cc:442] Loaded cuDNN version 8700
2023-10-23 00:00:45.415537: I ./tensorflow/compiler/jit/device_compiler.h:186] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


1202/1203 [============================>.] - ETA: 0s - loss: 3.6880 - accuracy: 0.306416
16

Epoch 1: val_accuracy improved from -inf to 0.35688, saving model to model.hdf5


/home/guille/anaconda3/envs/vision2/lib/python3.9/site-packages/keras/src/engine/training.py:3079: UserWarning: You are saving your model as an HDF5 file via `model.save()`. This file format is considered legacy. We recommend using instead the native Keras format, e.g. `model.save('my_model.keras')`.
  saving_api.save_model(


1203/1203 [==============================] - 73s 57ms/step - loss: 3.6876 - accuracy: 0.3064 - val_loss: 2.8428 - val_accuracy: 0.3569 - lr: 0.0100
Epoch 2/30
1199/1203 [============================>.] - ETA: 0s - loss: 2.6436 - accuracy: 0.3447

#### Report

You must prepare a report (PDF) describing:
* The problems and data sets (briefly).
* The process that you have followed to reach your solution for the “xview_recognition” benchmark, including your intermediate results. You must discuss and compare these results properly.
* Final network architectures, including optimization algorithms, regularization methods (dropout, data augmentation, etc.), number of layers/parameters, and performance obtained with your model on the train/valid/test data sets, including the plots of the evolution of losses and accuracy.
* It would also be very valuable your feedback on the use of “Cesvima” or “Google Colab" services.

In the submission via Moodle, attach your Python (.py) or Jupyter Notebook (.ipynb) source file, including in the report all results of computations attached to the code that generated them.

The assignment must be done in groups of 3 students.